# 03 - Évaluation du modèle (Personne 2)

**Objectif** : évaluer `best_model.keras` sur le jeu de **test** (jamais vu à l'entraînement) : accuracy, precision,
recall, F1-score, matrice de confusion et analyse des erreurs.

Prérequis : `models/best_model.keras` (Personne 1) et `dataset/test/` (`python src/prepare_dataset.py --raw data_raw`).

In [ ]:
import sys
from pathlib import Path

# permet d'importer le dossier src/ depuis le dossier notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             precision_recall_fscore_support)

from src.common import BEST_MODEL_PATH, CLASS_NAMES, RESULTS_DIR
from src.data_loader import load_split

RESULTS_DIR.mkdir(exist_ok=True)
model = keras.models.load_model(BEST_MODEL_PATH)
print("Modèle chargé :", BEST_MODEL_PATH.name, "| TensorFlow", tf.__version__)

## 1. Prédictions sur le jeu de test

In [ ]:
test_ds, test_paths = load_split("test", batch_size=32, shuffle=False, return_paths=True)

y_true = np.concatenate([y.numpy() for _, y in test_ds])
probs = model.predict(test_ds, verbose=0)
y_pred = probs.argmax(axis=1)
confidence = probs.max(axis=1)
print(f"{len(y_true)} images de test")

## 2. Métriques : Accuracy, Precision, Recall, F1-score

In [ ]:
acc = accuracy_score(y_true, y_pred)
p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
print(f"Accuracy         : {acc:.4f}")
print(f"Precision (macro): {p:.4f}")
print(f"Recall    (macro): {r:.4f}")
print(f"F1-score  (macro): {f1:.4f}\n")

report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4, zero_division=0)
print(report)

report_df = pd.DataFrame(classification_report(y_true, y_pred, target_names=CLASS_NAMES,
                                               output_dict=True, zero_division=0)).T
report_df.to_csv(RESULTS_DIR / "classification_report.csv")
report_df.round(4)

## 3. Matrice de confusion

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASS_NAMES)))
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title("Matrice de confusion (effectifs)")
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title("Matrice de confusion (normalisée par ligne)")
for ax in axes:
    ax.set_xlabel("Classe prédite"); ax.set_ylabel("Classe réelle")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=150)
plt.show()

## 4. Analyse des erreurs de classification

In [ ]:
errors = pd.DataFrame({
    "fichier": [p for p in test_paths],
    "classe_reelle": [CLASS_NAMES[i] for i in y_true],
    "classe_predite": [CLASS_NAMES[i] for i in y_pred],
    "confiance": confidence,
})
errors = errors[y_true != y_pred].sort_values("confiance", ascending=False)
errors.to_csv(RESULTS_DIR / "misclassified.csv", index=False)
print(f"{len(errors)} erreur(s) sur {len(y_true)} images ({len(errors) / len(y_true):.1%})")

if len(errors):
    pairs = (errors.groupby(["classe_reelle", "classe_predite"]).size()
             .sort_values(ascending=False).rename("nombre").reset_index())
    print("\nConfusions les plus fréquentes :")
    display(pairs.head(8))

In [ ]:
from PIL import Image

if len(errors):
    sel = errors.head(12)
    cols = 4
    rows = int(np.ceil(len(sel) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.4 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, (_, row) in zip(axes, sel.iterrows()):
        ax.imshow(Image.open(row["fichier"]).convert("L"), cmap="gray")
        ax.set_title(f"réel : {row['classe_reelle']}\nprédit : {row['classe_predite']} ({row['confiance']:.0%})",
                     fontsize=8, color="crimson")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "misclassified.png", dpi=130)
    plt.show()

## 5. Analyse et conclusions

*À rédiger par Personne 2 :*
- Quelles classes sont les mieux / les moins bien reconnues ?
- Quelles paires de classes sont confondues, et pourquoi (ressemblance visuelle, bruit, contraste) ?
- Le modèle est-il trop confiant sur ses erreurs ?
- Améliorations possibles (à transmettre à Personne 1) : plus d'augmentation, architecture plus profonde,
  MobileNetV2 en transfert d'apprentissage.